# V'DJer — реконструкция BCR из симулированных PE150 — мышь ERP003950 (IgG, только IGH)

Шесть образцов мыши, все IGH (IgG): каждый образец обрабатывается отдельно, цепь только IGH (IGK/IGL в датасете нет).

**Что делает V'DJer.** Программа рассчитана на RNA-seq, выровненный на геном: из BAM берутся риды около IG-локуса и невыровненные риды, по ним строится граф де Брёйна (k = 35), обход идёт от V-якорей к J-якорям, окно вокруг CDR3 проверяется по покрытию ридами. Цепь входит в алгоритм (якоря, индексы и координаты локуса свои для IGH, IGK, IGL), поэтому цепи запускаются раздельно, но на одном и том же FASTQ.

**Адаптация под наши ампликоны** (подробный разбор — `docs/audit_simulation_vdjer_truth_20260929.md`):

1. В исходниках V'DJer жёстко задано `MIN_CONTIG_SIZE 550` нт, а наши ампликоны имеют длину 380–500 нт, поэтому используется пересобранная копия `tools/vdjer-amplicon/vdjer` с `MIN_CONTIG_SIZE 350` (оригинальный `tools/vdjer-0.12` и `bcr_env/bin/vdjer` не изменены).
2. Геометрия ампликона (из истины симуляции ERR346600): до начала CDR3 286–295 нт; **после CDR3 ровно 63 нт**; длина junction 24–57 нт; длина ампликона 379–412 нт.
Для мыши подобрано `-jext 60`, `--ws 364`, окно 364 нт, `--wo 218`.
3. `--e0/--e1` задают и область проверки покрытия, и **длину выдаваемого контига** (`e1 − e0 + 1`).
4. Фильтр покрытия (`--rf`, `coverage.c`) засчитывает только риды, целиком и без ошибок совпадающие с окном, и требует (а) хотя бы `rf` ридов, начинающихся **до** `e0`, и (б) рид, начинающийся после `e1 − длина рида` и целиком лежащий в окне. Поэтому `e0 ≥ 2` и `e1 ≤ ws − 1`: при `--e0 1 --e1 ws` фильтр не пропускает ни одного контига при любой глубине, а `--rf 0` выдаёт непроверенные пути графа.

**Автоподбор по ридам.** Длину рида V'DJer берёт из BAM сам; `--ins` — медиана длины вставки по перекрытию R1 и R2 первых 200 тыс. пар (геномный TLEN завышен интроном лидер–V; запасное значение 250); если в утверждённых параметрах нет `e0/e1`, берутся `e0 = 20`, `e1 = ws − 1`. Геометрия окна (`ws`, `-jext`) определяется дизайном ампликона (праймерами) и задаётся в файле утверждения.

Готового мышиного набора у V'DJer нет, поэтому он строится из мышиных IMGT-аллелей (`tools/TRUST4/mouse/mouse_IMGT+C.fa`) и генома mm39 (chr12) скриптом `scripts/build_vdjer_bundle.py`: локус IGH GRCm39 `chr12:113218450-115973713` (обратная цепь), V-область `chr12:113490429-115973713`, константная/D `chr12:113218450-113490429`. Выравнивание STAR на `chr12` mm39.

**Параметры не зашиты в блокнот.** Их читает файл утверждения `vdjer_approved/mouse.json`, который создаётся только после положительного бенчмарка пилота на подвыборке. Если файла нет, блокнот останавливается до запуска: так его безопасно ставить в очередь заранее.

Выход для каждого образца: `<ветка>/assemblers/vdjer/<образец>/` — нативные файлы по цепям, `provenance.json` и нормализованный `contigs.fasta` для `benchmark_assemblers_mouse.ipynb`. Готовый результат не пересчитывается (`FORCE=False`).

## Окружение и параметры

In [ ]:
import os, sys, shutil, subprocess, time, gzip, json, hashlib, concurrent.futures
from pathlib import Path

BCR_ENV = Path(os.environ.get("BCR_ENV", "/data/user/epishkin/conda/envs/bcr_env"))
os.environ["PATH"] = str(BCR_ENV / "bin") + os.pathsep + os.environ["PATH"]

DATASET = "ERP003950"
SPECIES = "mouse"
CHAINS = ['IGH']
BRANCH = os.environ.get("BCR_BRANCH", "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut_amp_ec1x3_in5ng_rb3x_ss250")
SAMPLES = os.environ["BCR_SAMPLES"].split(",") if os.environ.get("BCR_SAMPLES") else ["ERR346596", "ERR346597", "ERR346598", "ERR346599", "ERR346600", "ERR346601"]
THREADS = int(os.environ.get("BCR_THREADS", 12))
SUBSAMPLE_PAIRS = int(os.environ["BCR_VDJER_SUBSAMPLE_PAIRS"]) if os.environ.get("BCR_VDJER_SUBSAMPLE_PAIRS") else None   # None = все пары
INSERT = 250            # запасное --ins; для каждого образца заменяется оценкой по ридам (estimate_insert)
FORCE = False

def find_root():
    cwd = Path.cwd().resolve()
    for root in (Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin")), cwd, *cwd.parents):
        if (root / "results" / DATASET / "simulated" / BRANCH).is_dir():
            return root
    raise FileNotFoundError(f"results/{DATASET}/simulated/{BRANCH} not found; set BCR_VOLUME")

ROOT = find_root()
DATASET_DIR = ROOT / "results" / DATASET
BRANCH_DIR = DATASET_DIR / "simulated" / BRANCH
FASTQ_DIR = BRANCH_DIR / "06_fastq_pe150"
ASM_DIR = BRANCH_DIR / "assemblers"
VDJER_BIN = ROOT / "tools" / "vdjer-amplicon" / "vdjer"
APPROVAL = ROOT / "vdjer_approved" / f"{SPECIES}.json"

def reads(sample):
    r1, r2 = FASTQ_DIR / f"{sample}_R1.fastq.gz", FASTQ_DIR / f"{sample}_R2.fastq.gz"
    if not (r1.exists() and r2.exists()):
        raise FileNotFoundError(f"simulated FASTQ missing for {sample}: {r1}")
    return r1, r2

def run_logged(cmd, log_path, cwd=None, heartbeat=300):
    log_path = Path(log_path); log_path.parent.mkdir(parents=True, exist_ok=True)
    print("RUN:", " ".join(map(str, cmd)), "\nLOG:", log_path, flush=True)
    t0, last = time.time(), time.time()
    with open(log_path, "w") as log:
        p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
        while p.poll() is None:
            time.sleep(10)
            if time.time() - last >= heartbeat:
                print(f"  running {(time.time() - t0) / 60:.0f} min", flush=True); last = time.time()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}; see {log_path}")
    print(f"done in {(time.time() - t0) / 60:.1f} min", flush=True)

def iter_fasta(path):
    name, chunks = None, []
    with (gzip.open(path, "rt") if str(path).endswith(".gz") else open(path)) as h:
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:].split()[0], []
            else:
                chunks.append(line.strip())
    if name is not None:
        yield name, "".join(chunks)

def write_contigs(src_pairs, out_fa, prefix):
    """Нормализованный FASTA контигов для сравнения с эталоном."""
    n = 0
    tmp = Path(str(out_fa) + ".tmp")
    with open(tmp, "w") as h:
        for name, seq in src_pairs:
            seq = seq.upper()
            if seq:
                n += 1
                h.write(f">{prefix}_{n} {name}\n{seq}\n")
    tmp.replace(out_fa)
    print(out_fa, n, "contigs")
    return n

def sha256(path, limit=None):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

# Гейт: без утверждённых параметров пилота блокнот не запускается
if not APPROVAL.exists():
    raise RuntimeError(f"Параметры V'DJer не утверждены: нет {APPROVAL}. Файл создаётся после положительного бенчмарка пилота.")
APPROVED = json.loads(APPROVAL.read_text())
assert APPROVED["species"] == SPECIES and all(ch in APPROVED["chains"] for ch in CHAINS)
assert VDJER_BIN.exists(), VDJER_BIN

print("ROOT:", ROOT); print("BRANCH:", BRANCH_DIR); print("samples:", SAMPLES, "threads:", THREADS, "subsample:", SUBSAMPLE_PAIRS)
print("утверждено:", APPROVAL, APPROVED.get("date"), "|", APPROVED.get("pilot_summary"))

## Референсы и выравнивание на геном

In [ ]:
REF_ROOT = ROOT / "seq_refs"
MM39 = REF_ROOT / "mm39"
STAR_INDEX = MM39 / "star_chr12"
BUNDLE = REF_ROOT / "vdjer_mouse" / "igh"               # строится из IMGT-аллелей мыши и генома mm39
BUILDER = ROOT / "tools" / "vdjer-amplicon" / "build_vdjer_bundle.py"
MOUSE_IMGT = ROOT / "tools" / "TRUST4" / "mouse" / "mouse_IMGT+C.fa"
IGH_LOCUS = "chr12:113218450-115973713"                 # весь локус IGH GRCm39 (обратная цепь)
IGH_C_REGION = "chr12:113218450-113490429"              # константная часть и D
IGH_V_REGION = "113490429-115973713"                    # V-область для v_region.fa

def chain_reference_args(chain):
    assert chain == "IGH"
    return ["--vr", IGH_LOCUS, "--cr", IGH_C_REGION, "--ref-dir", BUNDLE]   # после --chain

if not (BUNDLE / "provenance.json").exists():
    assert (MM39 / "chr12.fa.gz").exists(), f"нужен геном mm39 chr12: {MM39 / 'chr12.fa.gz'}"
    run_logged([sys.executable, BUILDER, "--imgt", MOUSE_IMGT, "--chain", "IGH", "--genome", MM39 / "chr12.fa.gz",
                "--contig", "chr12", "--v-region", IGH_V_REGION, "--reverse-strand", "--v-rule", "cys", "--j-rule", "fixed16",
                "--max-dist", "4", "--out", BUNDLE], BUNDLE.parent / "build_igh.log")
if not (STAR_INDEX / "SA").exists():
    fa = MM39 / "chr12.fa"
    if not fa.exists():
        with gzip.open(MM39 / "chr12.fa.gz", "rt") as h, open(fa, "w") as out:
            shutil.copyfileobj(h, out)
    STAR_INDEX.mkdir(parents=True, exist_ok=True)
    run_logged(["STAR", "--runMode", "genomeGenerate", "--runThreadN", THREADS, "--genomeDir", STAR_INDEX,
                "--genomeFastaFiles", fa, "--genomeSAindexNbases", 12, "--outFileNamePrefix", str(STAR_INDEX) + "/"], STAR_INDEX / "build.log")
print("V'DJer bundle:", BUNDLE); print("STAR index:", STAR_INDEX)

## Запуск V'DJer

In [ ]:
def subsample_pairs(r1, r2, out1, out2, n_pairs):
    """Детерминированная подвыборка n_pairs пар (seed 42) — только если задан BCR_VDJER_SUBSAMPLE_PAIRS."""
    import random
    with gzip.open(r1, "rt") as h:
        total = sum(1 for _ in h) // 4
    p = min(1.0, n_pairs / total)
    rng = random.Random(42)
    kept = 0
    with gzip.open(r1, "rt") as f1, gzip.open(r2, "rt") as f2, open(out1, "w") as o1, open(out2, "w") as o2:
        while True:
            a = [f1.readline() for _ in range(4)]
            b = [f2.readline() for _ in range(4)]
            if not a[0]:
                break
            if rng.random() < p:
                o1.writelines(a); o2.writelines(b); kept += 1
    print(f"подвыборка: {kept:,} пар из {total:,}", flush=True)
    return total

def align(sample, out):
    """STAR: сортированный индексированный BAM; невыровненные риды сохраняются (--outSAMunmapped Within), как требует V'DJer."""
    bam = out / "star.sort.bam"
    if Path(str(bam) + ".bai").exists() and not FORCE:
        return bam
    r1, r2 = reads(sample)
    fq = ["--readFilesIn", r1, r2, "--readFilesCommand", "zcat"]
    if SUBSAMPLE_PAIRS:
        s1, s2 = out / "sub_R1.fastq", out / "sub_R2.fastq"
        subsample_pairs(r1, r2, s1, s2, SUBSAMPLE_PAIRS)
        fq = ["--readFilesIn", s1, s2]
    run_logged(["STAR", "--runThreadN", THREADS, "--genomeDir", STAR_INDEX] + fq +
               ["--outSAMtype", "BAM", "SortedByCoordinate", "--outSAMunmapped", "Within",
                "--limitBAMsortRAM", 20_000_000_000, "--outFileNamePrefix", str(out / "star_")], out / "star.log")
    (out / "star_Aligned.sortedByCoord.out.bam").rename(bam)
    run_logged(["samtools", "index", bam], out / "index.log")
    return bam

def estimate_insert(r1, r2, n=200_000, seed_len=20, fallback=250):
    """Медиана длины вставки по самим ридам: начало revcomp(R2) ищется в R1 (перекрытие >= seed_len нт).
    Пары без перекрытия (вставка > 2*длина рида - seed_len) считаются длинными; медиана берётся по всем парам.
    По геномному BAM оценивать нельзя: интрон лидер–V и сплайсинг V–J завышают TLEN."""
    comp = str.maketrans("ACGTN", "TGCAN")
    found, total = [], 0
    with gzip.open(r1, "rt") as f1, gzip.open(r2, "rt") as f2:
        while total < n:
            a = [f1.readline() for _ in range(4)]; b = [f2.readline() for _ in range(4)]
            if not a[0]:
                break
            s1, s2 = a[1].strip(), b[1].strip().translate(comp)[::-1]
            total += 1
            o = s1.find(s2[:seed_len])
            if o >= 0:
                found.append(o + len(s2))
    found.sort()
    k = total // 2
    return found[k] if total >= 1000 and k < len(found) else fallback

def vdjer_command(chain, bam, threads):
    prm = dict(APPROVED["chains"][chain]["params"])
    prm.setdefault("e0", 20)                       # coverage.c: нужен рид, начинающийся до e0
    prm.setdefault("e1", int(prm["ws"]) - 1)       # и рид, начинающийся после e1 - длина рида, целиком в окне
    cmd = [VDJER_BIN, "--in", bam, "--t", threads, "--ins", INSERT, "--chain", chain] + chain_reference_args(chain)
    for k, v in prm.items():                      # jext, ws, e0, e1, wo, rf, ... как утверждено пилотом
        cmd += [("-" if k == "jext" else "--") + k, v]
    return [str(x) for x in cmd]

def run_chain(chain, bam, out, threads):
    cdir = out / chain
    cdir.mkdir(exist_ok=True)
    exit_json = cdir / "exit.json"
    if not exit_json.exists() or FORCE:
        cmd = vdjer_command(chain, bam, threads)
        t0 = time.time()
        with open(cdir / "vdjer.sam", "w") as sam, open(cdir / "vdjer.log", "w") as log:
            rc = subprocess.run(cmd, cwd=cdir, stdout=sam, stderr=log).returncode
        exit_json.write_text(json.dumps({"command": cmd, "returncode": rc, "seconds": round(time.time() - t0)}))
    info = json.loads(exit_json.read_text())
    if info["returncode"]:
        raise RuntimeError(f"V'DJer {chain} exit {info['returncode']}; see {cdir / 'vdjer.log'}")
    return chain

def reads_per_contig(sam):
    counts = {}
    with open(sam) as h:
        for line in h:
            if line[0] == "@":
                continue
            rname = line.split("\t", 3)[2]
            if rname != "*":
                counts[rname] = counts.get(rname, 0) + 1
    return counts

for sample in SAMPLES:
    out = ASM_DIR / "vdjer" / sample
    contigs = out / "contigs.fasta"
    if contigs.exists() and not FORCE:
        print("[skip]", sample); continue
    out.mkdir(parents=True, exist_ok=True)
    bam = align(sample, out)
    INSERT = estimate_insert(*reads(sample))
    print("--ins (медиана вставки по перекрытию R1/R2):", INSERT, flush=True)
    threads = max(1, THREADS // len(CHAINS))
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(CHAINS)) as pool:
        list(pool.map(lambda ch: run_chain(ch, bam, out, threads), CHAINS))
    pairs, per_chain = [], {}
    for chain in CHAINS:
        cdir = out / chain
        fa = cdir / "vdj_contigs.fa"
        found = list(iter_fasta(fa)) if fa.exists() else []
        support = reads_per_contig(cdir / "vdjer.sam")
        min_reads = APPROVED["chains"][chain].get("min_reads", 0)
        kept = [(f"{chain}_{n}", s) for n, s in found if support.get(n, 0) >= min_reads]
        per_chain[chain] = {"contigs_found": len(found), "contigs_kept": len(kept), "min_reads": min_reads}
        pairs += kept
    n = write_contigs(pairs, contigs, "vdjer")
    (out / "provenance.json").write_text(json.dumps({
        "sample": sample, "branch": BRANCH, "species": SPECIES, "subsample_pairs": SUBSAMPLE_PAIRS,
        "binary": str(VDJER_BIN), "binary_sha256": sha256(VDJER_BIN),
        "approval": str(APPROVAL), "approval_sha256": sha256(APPROVAL), "insert_estimated": INSERT, "chains": per_chain, "contigs_total": n}, indent=1))